# Classification NBA Model

## Configuration

## Imports

In [47]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training,
)
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
)
from sklearn.model_selection import cross_validate
from xgboost import XGBRegressor


In [48]:
from nba_ou.modeling.modeling import (
    TemporalDecaySampleWeightRegressor,
    assert_valid_time_splits,
    build_recency_sample_weights,
    evaluate_day_by_day_walk_forward,
    make_test_anchored_walk_forward_splits,
    save_model_bundle,
    load_model_bundle,
    split_latest_dates_holdout,
)


In [49]:
nan_threshold = 50.0
max_na_per_row = 70

## Load Data

In [50]:
exclude = "fanatics_sportsbook"

In [51]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260408.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [52]:
df_stats = df_stats[df_stats['SEASON_YEAR'] >= 2021]

In [53]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=nan_threshold, max_na_per_row=max_na_per_row, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'], exclude_cols_containing=[exclude])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 6478 rows
Basic cleaning complete: 6463 rows remaining

Starting advanced column cleaning with 2948 columns

Advanced column cleaning complete: 2948 → 2052 columns (896 removed)


Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 1 (0.02%)
  Critical columns requiring data: 4
  Columns zero-filled: 112
  Infer pairs applied: 0/106
  Remaining NaN cells: 251178

Dropping rows with more than 70 NaN values...
Removed 663 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (5799, 2052)


In [54]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_to_train.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1730,total_consensus_pct_under_TREND_SLOPE_LAST_5_H...,761,13.12,2023.0
1728,total_consensus_pct_over_TREND_SLOPE_LAST_5_HO...,748,12.90,2023.0
1734,spread_consensus_pct_home_TREND_SLOPE_LAST_5_H...,678,11.69,2023.0
1732,spread_consensus_pct_away_TREND_SLOPE_LAST_5_H...,667,11.50,2023.0
1729,total_consensus_pct_under_TREND_SLOPE_LAST_5_G...,654,11.28,2023.0
...,...,...,...,...
1884,spread_betmgm_price_home,1,0.02,2021.0
1873,total_betmgm_price_under,1,0.02,2021.0
1958,odds_ml_home_prob_novig_betmgm,1,0.02,2025.0
1959,odds_ml_vig_betmgm,1,0.02,2025.0


In [55]:
BET365_LINE_COL =  "TOTAL_LINE_bet365"
# BET365_LINE_COL =  "total_bet365_line_over"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [56]:
df_to_train['GAME_DATE'] = pd.to_datetime(df_to_train['GAME_DATE'])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

In [57]:
#count games per season
games_per_season = df_to_train.groupby('SEASON_YEAR').size()
print(games_per_season)

SEASON_YEAR
2021    1239
2022    1233
2023     967
2024    1238
2025    1122
dtype: int64


## Train / Test

In [58]:
TARGET_COL = "TOTAL_POINTS"
SAMPLE_WEIGHT_LAMBDA = 0.0075
SAMPLE_WEIGHT_LAMBDA_BOUNDS = (1e-4, 0.015)
TRAIN_GAMES = 2500
DAY_BY_DAY_METRIC_NAME = "OU_Betting_Accuracy"
DAY_BY_DAY_THRESHOLDS = (1, 2, 3)
BET_STAKE_EUR = 1.0
BET_DECIMAL_ODDS = 1.90
BET_WIN_NET_PROFIT_EUR = BET_STAKE_EUR * (BET_DECIMAL_ODDS - 1.0)


In [59]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.05,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print("Final test date range:",
      df_test_final["GAME_DATE"].min(), "->", df_test_final["GAME_DATE"].max())

Development set size: 5501
Final test set size: 298
Final test date range: 2026-03-01 00:00:00 -> 2026-04-08 00:00:00


In [60]:
EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")
sample_weight_dev = build_recency_sample_weights(
    df_dev,
    lambda_=SAMPLE_WEIGHT_LAMBDA,
)

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")
print(
    f"Recency sample weights lambda={SAMPLE_WEIGHT_LAMBDA}: "
    f"min={sample_weight_dev.min():.4f}, max={sample_weight_dev.max():.4f}"
)


X_dev shape: (5501, 2049)
X_test_final shape: (298, 2049)
Recency sample weights lambda=0.0075: min=0.0000, max=1.0000


In [61]:
from nba_ou.modeling.scorers import (
    OverUnderScorerTotalPoints,
    OverUnderScorerTotalPointsMinEdge,
    evaluate_total_points_thresholds,
    over_under_betting_accuracy_total_points,
    over_under_betting_accuracy_total_points_with_min_edge,
)

ou_scorer = OverUnderScorerTotalPoints(BET365_LINE_COL)
ou_scorer_edge_2 = OverUnderScorerTotalPointsMinEdge(
    line_col=BET365_LINE_COL,
    min_edge=2,
)
ou_scorer_edge_4 = OverUnderScorerTotalPointsMinEdge(
    line_col=BET365_LINE_COL,
    min_edge=4,
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
    "OU_Betting_Accuracy_Edge_2": ou_scorer_edge_2,
    "OU_Betting_Accuracy_Edge_4": ou_scorer_edge_4,
}


def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_vals = cv_results[train_key]
        test_vals = cv_results[test_key]

        train_val = np.nanmean(train_vals)
        test_val = np.nanmean(test_vals)

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if "OU_Betting_Accuracy" in sc:
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")
            n_valid = np.sum(~np.isnan(test_vals))
            print(f"  (valid folds: {n_valid}/{len(test_vals)})")
        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")
        print()


def calculate_flat_bet_profit_total_points(
    y_true,
    y_pred,
    betting_line,
    *,
    stake=BET_STAKE_EUR,
    win_net_profit=BET_WIN_NET_PROFIT_EUR,
):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    betting_line = np.asarray(betting_line, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred) & np.isfinite(betting_line)
    true_side = np.sign(y_true - betting_line)
    pred_side = np.sign(y_pred - betting_line)
    bet_mask = valid & (pred_side != 0)

    profits = np.zeros(len(y_true), dtype=float)
    wins = bet_mask & (true_side == pred_side)
    losses = bet_mask & (true_side != 0) & (true_side != pred_side)
    pushes = bet_mask & (true_side == 0)

    profits[wins] = win_net_profit
    profits[losses] = -stake
    profits[pushes] = 0.0

    return {
        "n_games": int(valid.sum()),
        "n_bets": int(bet_mask.sum()),
        "n_wins": int(wins.sum()),
        "n_losses": int(losses.sum()),
        "n_pushes": int(pushes.sum()),
        "total_profit_eur": float(profits.sum()),
        "avg_profit_per_game_eur": float(profits.sum() / valid.sum()) if valid.sum() else np.nan,
        "avg_profit_per_bet_eur": float(profits.sum() / bet_mask.sum()) if bet_mask.sum() else np.nan,
        "roi_per_bet": float(profits.sum() / (stake * bet_mask.sum())) if bet_mask.sum() else np.nan,
    }


def summarize_walk_forward_total_points(
    predictions_df,
    daily_template,
    df_test_final,
    *,
    line_col=BET365_LINE_COL,
    metric_name=DAY_BY_DAY_METRIC_NAME,
    thresholds=DAY_BY_DAY_THRESHOLDS,
):
    line_lookup = (
        df_test_final.reset_index(drop=True)[[line_col]]
        .reset_index()
        .rename(columns={"index": "row_in_test_final"})
    )

    scored_predictions = predictions_df.merge(
        line_lookup,
        on="row_in_test_final",
        how="left",
        validate="many_to_one",
    ).copy()

    scored_predictions["date"] = pd.to_datetime(
        scored_predictions["date"],
        errors="coerce",
    ).dt.normalize()
    scored_predictions["y_true"] = pd.to_numeric(
        scored_predictions["y_true"],
        errors="coerce",
    )
    scored_predictions["y_pred"] = pd.to_numeric(
        scored_predictions["y_pred"],
        errors="coerce",
    )
    scored_predictions[line_col] = pd.to_numeric(
        scored_predictions[line_col],
        errors="coerce",
    )

    daily_context = daily_template.copy()
    daily_context["date"] = pd.to_datetime(
        daily_context["date"],
        errors="coerce",
    ).dt.normalize()
    daily_context = daily_context.drop(columns=["_walk_mae"], errors="ignore")

    daily_rows = []
    for current_day, day_df in scored_predictions.groupby("date", sort=True):
        context_row = daily_context.loc[daily_context["date"] == current_day].iloc[0].to_dict()
        y_true_day = day_df["y_true"].to_numpy(dtype=float)
        y_pred_day = day_df["y_pred"].to_numpy(dtype=float)
        betting_line_day = day_df[line_col].to_numpy(dtype=float)
        profit_day = calculate_flat_bet_profit_total_points(
            y_true=y_true_day,
            y_pred=y_pred_day,
            betting_line=betting_line_day,
        )

        context_row[metric_name] = over_under_betting_accuracy_total_points(
            y_true=y_true_day,
            y_pred=y_pred_day,
            betting_line=betting_line_day,
        )
        context_row["bet_profit_eur"] = profit_day["total_profit_eur"]
        context_row["avg_profit_per_game_eur"] = profit_day["avg_profit_per_game_eur"]
        context_row["avg_profit_per_bet_eur"] = profit_day["avg_profit_per_bet_eur"]
        context_row["bet_roi"] = profit_day["roi_per_bet"]
        context_row["n_bets"] = profit_day["n_bets"]
        context_row["n_wins"] = profit_day["n_wins"]
        context_row["n_losses"] = profit_day["n_losses"]
        context_row["n_pushes"] = profit_day["n_pushes"]
        daily_rows.append(context_row)

    daily_results = pd.DataFrame(daily_rows)

    y_true = scored_predictions["y_true"].to_numpy(dtype=float)
    y_pred = scored_predictions["y_pred"].to_numpy(dtype=float)
    betting_line = scored_predictions[line_col].to_numpy(dtype=float)
    pred_edge = y_pred - betting_line
    margin = np.abs(pred_edge)
    n_total = len(scored_predictions)
    overall_profit = calculate_flat_bet_profit_total_points(
        y_true=y_true,
        y_pred=y_pred,
        betting_line=betting_line,
    )

    threshold_rows = []
    for threshold in thresholds:
        mask = margin > threshold
        n_games = int(mask.sum())
        threshold_profit = calculate_flat_bet_profit_total_points(
            y_true=y_true[mask],
            y_pred=y_pred[mask],
            betting_line=betting_line[mask],
        )
        ou_acc = (
            np.nan
            if n_games == 0
            else over_under_betting_accuracy_total_points(
                y_true=y_true[mask],
                y_pred=y_pred[mask],
                betting_line=betting_line[mask],
            )
        )
        threshold_rows.append(
            {
                "threshold_abs_pred_edge_gt": threshold,
                "n_games": n_games,
                "pct_of_test": (n_games / n_total) if n_total else np.nan,
                "ou_betting_accuracy": ou_acc,
                "bet_profit_eur": threshold_profit["total_profit_eur"],
                "avg_profit_per_game_eur": threshold_profit["avg_profit_per_game_eur"],
                "avg_profit_per_bet_eur": threshold_profit["avg_profit_per_bet_eur"],
                "bet_roi": threshold_profit["roi_per_bet"],
                "n_bets": threshold_profit["n_bets"],
            }
        )

    threshold_results = pd.DataFrame(threshold_rows)
    return scored_predictions, daily_results, threshold_results, overall_profit


def run_day_by_day_walk_forward_evaluation(
    *,
    label,
    df_dev,
    df_test_final,
    fit_and_predict,
    max_games=TRAIN_GAMES,
    metric_name=DAY_BY_DAY_METRIC_NAME,
    thresholds=DAY_BY_DAY_THRESHOLDS,
):
    raw_result = evaluate_day_by_day_walk_forward(
        df_dev=df_dev,
        df_test_final=df_test_final,
        fit_and_predict=fit_and_predict,
        metric_fn=lambda y_true, y_pred: mean_absolute_error(y_true, y_pred),
        target_col=TARGET_COL,
        max_games=max_games,
        metric_name="_walk_mae",
    )

    scored_predictions, daily_results, threshold_results, overall_profit = summarize_walk_forward_total_points(
        predictions_df=raw_result.predictions,
        daily_template=raw_result.daily_results,
        df_test_final=df_test_final,
        line_col=BET365_LINE_COL,
        metric_name=metric_name,
        thresholds=thresholds,
    )

    mean_metric = float(daily_results[metric_name].mean())
    print(f"{label} mean day-by-day {metric_name}: {mean_metric:.2%}")
    print(
        f"{label} flat-bet profit at {BET_DECIMAL_ODDS:.2f} odds: "
        f"{overall_profit['total_profit_eur']:.2f} EUR total, "
        f"{overall_profit['avg_profit_per_game_eur']:.3f} EUR/game, "
        f"ROI {overall_profit['roi_per_bet']:.2%} over {overall_profit['n_bets']} bets"
    )
    display(
        daily_results.style.format(
            {
                metric_name: "{:.2%}",
                "bet_profit_eur": "{:.2f}",
                "avg_profit_per_game_eur": "{:.3f}",
                "avg_profit_per_bet_eur": "{:.3f}",
                "bet_roi": "{:.2%}",
            }
        )
    )
    print(f"{label} thresholded walk-forward accuracy")
    display(
        threshold_results.style.format(
            {
                "pct_of_test": "{:.1%}",
                "ou_betting_accuracy": "{:.2%}",
                "bet_profit_eur": "{:.2f}",
                "avg_profit_per_game_eur": "{:.3f}",
                "avg_profit_per_bet_eur": "{:.3f}",
                "bet_roi": "{:.2%}",
            }
        )
    )
    return raw_result, scored_predictions, daily_results, threshold_results, overall_profit


In [62]:
splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    test_games=50,
    step_games_between_tests=30,
    train_games=TRAIN_GAMES,
    min_train_games=int(TRAIN_GAMES * 0.5),
    max_folds=12,
    verbose=1,
)

assert_valid_time_splits(df_dev, splits)


Created 12 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           2500            51       2023-01-25     2025-03-17      2025-03-18    2025-03-24         2024
    2           2500            53       2023-02-06     2025-03-29      2025-03-30    2025-04-05         2024
    3           2500            51       2023-02-24     2025-04-09      2025-04-10    2025-04-25         2024
    4           2500            50       2023-03-11     2025-06-22      2025-10-27    2025-11-06         2025
    5           2500            51       2023-03-24     2025-11-10      2025-11-11    2025-11-17         2025
    6           2500            57       2023-04-04     2025-11-22      2025-11-23    2025-11-30         2025
    7           2500            61       2023-05-07     2025-12-05      2025-12-06    2025-12-18         2025
    8           2500            53       2023-11-14     2025-12-23      2025

In [63]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)

DummyRegressor baseline
Train MAE: 15.58369
Validation MAE: 15.28953

Train RMSE: 19.45556
Validation RMSE: 18.97615

Train R2: 0.00000
Validation R2: -0.07201

Train OU_Betting_Accuracy: 49.28%
Validation OU_Betting_Accuracy: 49.55%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_2: 50.08%
Validation OU_Betting_Accuracy_Edge_2: 51.04%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_4: 50.19%
Validation OU_Betting_Accuracy_Edge_4: 50.17%
  (valid folds: 12/12)



In [64]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),   # LR cannot handle NaNs
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)

Linear Regression
Train MAE: 8.23541
Validation MAE: 53.59002

Train RMSE: 10.54939
Validation RMSE: 140.16734

Train R2: 0.70597
Validation R2: -327.58064

Train OU_Betting_Accuracy: 79.51%
Validation OU_Betting_Accuracy: 50.87%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_2: 82.87%
Validation OU_Betting_Accuracy_Edge_2: 51.98%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_4: 85.86%
Validation OU_Betting_Accuracy_Edge_4: 52.04%
  (valid folds: 12/12)



In [65]:
xgb_reg_no_weights = XGBRegressor(
    max_depth=4,
    learning_rate=0.057,
    n_estimators=75,
    subsample=0.8,
    colsample_bytree=0.86,
    reg_alpha=0.57,
    reg_lambda=1.78,
    min_child_weight=5.48,
    gamma=1.77,
    n_jobs=-1,
    random_state=16,
)

cv_results_no_weights = cross_validate(
    xgb_reg_no_weights,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("XGBoost no sample weights")
print_metrics(cv_results_no_weights)


XGBoost no sample weights
Train MAE: 10.17119
Validation MAE: 13.97351

Train RMSE: 12.75323
Validation RMSE: 17.33278

Train R2: 0.57029
Validation R2: 0.10414

Train OU_Betting_Accuracy: 83.17%
Validation OU_Betting_Accuracy: 51.92%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_2: 92.53%
Validation OU_Betting_Accuracy_Edge_2: 55.09%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_4: 97.46%
Validation OU_Betting_Accuracy_Edge_4: 58.05%
  (valid folds: 12/12)



In [66]:
xgb_reg_no_weights.fit(X_dev, y_dev)

y_pred_test_total = xgb_reg_no_weights.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)
ou_acc_edge_2 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")


Final test metrics
MSE: 317.27158
RMSE: 17.81212
MAE: 13.88581
OU_Betting_Accuracy: 50.85%
OU_Betting_Accuracy_Edge_2: 57.26%
OU_Betting_Accuracy_Edge_4: 48.39%


In [67]:
results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=xgb_reg_no_weights,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)


def fit_and_predict_xgb_no_weights_day_by_day(train_df, test_df):
    model = XGBRegressor(**xgb_reg_no_weights.get_params())

    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model.fit(X_train, y_train)
    return model.predict(X_test)


day_by_day_no_weights, day_by_day_no_weights_predictions, day_by_day_no_weights_daily, day_by_day_no_weights_thresholds, day_by_day_no_weights_profit = run_day_by_day_walk_forward_evaluation(
    label="XGBoost no sample weights",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_no_weights_day_by_day,
    max_games=TRAIN_GAMES,
)


,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,298,100.0%,50.85%
1,1,202,67.8%,51.52%
2,2,120,40.3%,57.26%
3,3,65,21.8%,51.56%
4,4,31,10.4%,48.39%
5,5,16,5.4%,56.25%
6,6,9,3.0%,55.56%
7,7,4,1.3%,75.00%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


XGBoost no sample weights mean day-by-day OU_Betting_Accuracy: 49.13%
XGBoost no sample weights flat-bet profit at 1.90 odds: -17.50 EUR total, -0.059 EUR/game, ROI -5.87% over 298 bets


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy,bet_profit_eur,avg_profit_per_game_eur,avg_profit_per_bet_eur,bet_roi,n_bets,n_wins,n_losses,n_pushes
0,2026-03-01 00:00:00,2500,11,2024-02-25 00:00:00,2026-02-28 00:00:00,27.27%,-5.30,-0.482,-0.482,-48.18%,11,3,8,0
1,2026-03-02 00:00:00,2500,4,2024-02-27 00:00:00,2026-03-01 00:00:00,75.00%,1.70,0.425,0.425,42.50%,4,3,1,0
2,2026-03-03 00:00:00,2500,10,2024-02-27 00:00:00,2026-03-02 00:00:00,70.00%,3.30,0.330,0.330,33.00%,10,7,3,0
3,2026-03-04 00:00:00,2500,6,2024-02-28 00:00:00,2026-03-03 00:00:00,33.33%,-2.20,-0.367,-0.367,-36.67%,6,2,4,0
4,2026-03-05 00:00:00,2500,9,2024-02-29 00:00:00,2026-03-04 00:00:00,22.22%,-5.20,-0.578,-0.578,-57.78%,9,2,7,0
5,2026-03-06 00:00:00,2500,7,2024-03-01 00:00:00,2026-03-05 00:00:00,57.14%,0.60,0.086,0.086,8.57%,7,4,3,0
6,2026-03-07 00:00:00,2500,6,2024-03-02 00:00:00,2026-03-06 00:00:00,33.33%,-2.20,-0.367,-0.367,-36.67%,6,2,4,0
7,2026-03-08 00:00:00,2500,10,2024-03-03 00:00:00,2026-03-07 00:00:00,62.50%,1.50,0.150,0.150,15.00%,10,5,3,2
8,2026-03-09 00:00:00,2500,5,2024-03-05 00:00:00,2026-03-08 00:00:00,60.00%,0.70,0.140,0.140,14.00%,5,3,2,0
9,2026-03-10 00:00:00,2500,11,2024-03-05 00:00:00,2026-03-09 00:00:00,45.45%,-1.50,-0.136,-0.136,-13.64%,11,5,6,0


XGBoost no sample weights thresholded walk-forward accuracy


,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy,bet_profit_eur,avg_profit_per_game_eur,avg_profit_per_bet_eur,bet_roi,n_bets
0,1,223,74.8%,49.77%,-11.90,-0.053,-0.053,-5.34%,223
1,2,148,49.7%,53.10%,1.30,0.009,0.009,0.88%,148
2,3,90,30.2%,52.81%,0.30,0.003,0.003,0.33%,90


In [68]:
xgb_reg_weights = XGBRegressor(
    max_depth=4,
    learning_rate=0.057,
    n_estimators=75,
    subsample=0.8,
    colsample_bytree=0.86,
    reg_alpha=0.57,
    reg_lambda=1.78,
    min_child_weight=5.48,
    gamma=1.77,
    n_jobs=-1,
    random_state=16,
)

weighted_xgb = TemporalDecaySampleWeightRegressor(
    estimator=xgb_reg_weights,
    dates=df_dev["GAME_DATE"],
    lambda_=SAMPLE_WEIGHT_LAMBDA,
)

cv_results_weights = cross_validate(
    weighted_xgb,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("XGBoost with sample weights (per-fold decay)")
print_metrics(cv_results_weights)


XGBoost with sample weights (per-fold decay)
Train MAE: 11.40245
Validation MAE: 14.15186

Train RMSE: 14.66015
Validation RMSE: 17.51638

Train R2: 0.43195
Validation R2: 0.08503

Train OU_Betting_Accuracy: 68.97%
Validation OU_Betting_Accuracy: 52.15%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_2: 75.90%
Validation OU_Betting_Accuracy_Edge_2: 51.19%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_4: 82.11%
Validation OU_Betting_Accuracy_Edge_4: 50.70%
  (valid folds: 12/12)



In [69]:
weighted_xgb.fit(X_dev, y_dev)

y_pred_test_total = weighted_xgb.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)
ou_acc_edge_2 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")


Final test metrics
MSE: 327.80066
RMSE: 18.10527
MAE: 14.10888
OU_Betting_Accuracy: 49.83%
OU_Betting_Accuracy_Edge_2: 50.00%
OU_Betting_Accuracy_Edge_4: 52.34%


In [70]:
results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=weighted_xgb,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)


def fit_and_predict_xgb_weights_day_by_day(train_df, test_df):
    base_model = XGBRegressor(**xgb_reg_weights.get_params())
    model = TemporalDecaySampleWeightRegressor(
        estimator=base_model,
        dates=train_df["GAME_DATE"],
        lambda_=SAMPLE_WEIGHT_LAMBDA,
    )

    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model.fit(X_train, y_train)
    return model.predict(X_test)


day_by_day_weights, day_by_day_weights_predictions, day_by_day_weights_daily, day_by_day_weights_thresholds, day_by_day_weights_profit = run_day_by_day_walk_forward_evaluation(
    label="XGBoost with sample weights",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_weights_day_by_day,
    max_games=TRAIN_GAMES,
)


,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,298,100.0%,49.83%
1,1,251,84.2%,49.19%
2,2,197,66.1%,50.00%
3,3,150,50.3%,52.35%
4,4,108,36.2%,52.34%
5,5,79,26.5%,56.41%
6,6,44,14.8%,50.00%
7,7,31,10.4%,45.16%
8,8,16,5.4%,37.50%
9,9,6,2.0%,16.67%


XGBoost with sample weights mean day-by-day OU_Betting_Accuracy: 52.33%
XGBoost with sample weights flat-bet profit at 1.90 odds: -0.40 EUR total, -0.001 EUR/game, ROI -0.13% over 298 bets


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy,bet_profit_eur,avg_profit_per_game_eur,avg_profit_per_bet_eur,bet_roi,n_bets,n_wins,n_losses,n_pushes
0,2026-03-01 00:00:00,2500,11,2024-02-25 00:00:00,2026-02-28 00:00:00,27.27%,-5.30,-0.482,-0.482,-48.18%,11,3,8,0
1,2026-03-02 00:00:00,2500,4,2024-02-27 00:00:00,2026-03-01 00:00:00,50.00%,-0.20,-0.050,-0.050,-5.00%,4,2,2,0
2,2026-03-03 00:00:00,2500,10,2024-02-27 00:00:00,2026-03-02 00:00:00,60.00%,1.40,0.140,0.140,14.00%,10,6,4,0
3,2026-03-04 00:00:00,2500,6,2024-02-28 00:00:00,2026-03-03 00:00:00,16.67%,-4.10,-0.683,-0.683,-68.33%,6,1,5,0
4,2026-03-05 00:00:00,2500,9,2024-02-29 00:00:00,2026-03-04 00:00:00,66.67%,2.40,0.267,0.267,26.67%,9,6,3,0
5,2026-03-06 00:00:00,2500,7,2024-03-01 00:00:00,2026-03-05 00:00:00,57.14%,0.60,0.086,0.086,8.57%,7,4,3,0
6,2026-03-07 00:00:00,2500,6,2024-03-02 00:00:00,2026-03-06 00:00:00,66.67%,1.60,0.267,0.267,26.67%,6,4,2,0
7,2026-03-08 00:00:00,2500,10,2024-03-03 00:00:00,2026-03-07 00:00:00,75.00%,3.40,0.340,0.340,34.00%,10,6,2,2
8,2026-03-09 00:00:00,2500,5,2024-03-05 00:00:00,2026-03-08 00:00:00,80.00%,2.60,0.520,0.520,52.00%,5,4,1,0
9,2026-03-10 00:00:00,2500,11,2024-03-05 00:00:00,2026-03-09 00:00:00,54.55%,0.40,0.036,0.036,3.64%,11,6,5,0


XGBoost with sample weights thresholded walk-forward accuracy


,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy,bet_profit_eur,avg_profit_per_game_eur,avg_profit_per_bet_eur,bet_roi,n_bets
0,1,241,80.9%,53.78%,5.20,0.022,0.022,2.16%,241
1,2,181,60.7%,53.93%,4.40,0.024,0.024,2.43%,181
2,3,136,45.6%,50.75%,-4.80,-0.035,-0.035,-3.53%,136


# OPTUNA

In [44]:
from nba_ou.modeling.optuna_total_points import (
    fit_best_xgb_total_points,
    select_best_trial_lexicographic,
    summarize_lexicographic_candidates,
    summarize_optuna_trials,
    tune_xgb_total_points_optuna,
)

study = tune_xgb_total_points_optuna(
    X=X_dev,
    y=y_dev,
    sample_weight_dates=df_dev["GAME_DATE"],
    tune_sample_weight_lambda=True,
    sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    splits=splits,
    line_col=BET365_LINE_COL,
    n_trials=80,
    timeout=4.5 * 3600,
    objective_name="reg:squarederror",
    study_name="xgb_total_points_mae",
)

best_trial_lexi = select_best_trial_lexicographic(
    study,
    mae_tolerance_abs=0.05,
)

print("Optuna best by MAE only")
print("Trial:", study.best_trial.number)
print("Best CV MAE:", study.best_value)
print("Mean OU accuracy:", study.best_trial.user_attrs.get("mean_ou_acc"))
print("Mean OU accuracy edge 2:", study.best_trial.user_attrs.get("mean_ou_acc_edge_2"))
print("Mean OU accuracy edge 4:", study.best_trial.user_attrs.get("mean_ou_acc_edge_4"))
print("Sample weight lambda:", study.best_trial.user_attrs.get("sample_weight_lambda"))

print()
print("Selected trial after MAE-first / OU-second ranking")
print("Trial:", best_trial_lexi.number)
print("CV MAE:", best_trial_lexi.user_attrs.get("mean_mae", best_trial_lexi.value))
print("Mean RMSE:", best_trial_lexi.user_attrs.get("mean_rmse"))
print("Mean R2:", best_trial_lexi.user_attrs.get("mean_r2"))
print("Mean OU accuracy:", best_trial_lexi.user_attrs.get("mean_ou_acc"))
print("Mean OU accuracy edge 2:", best_trial_lexi.user_attrs.get("mean_ou_acc_edge_2"))
print("Mean OU accuracy edge 4:", best_trial_lexi.user_attrs.get("mean_ou_acc_edge_4"))
print("Median best_iteration:", best_trial_lexi.user_attrs.get("median_best_iteration"))
print("Sample weight lambda:", best_trial_lexi.params.get("sample_weight_lambda"))
print("Params:")
for k, v in best_trial_lexi.params.items():
    print(f"{k}: {v}")

trials_df = summarize_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
            "mean_ou_acc_edge_2": "{:.2%}",
            "mean_ou_acc_edge_3": "{:.2%}",
            "mean_ou_acc_edge_4": "{:.2%}",
        }
    )
)

candidates_df = summarize_lexicographic_candidates(
    study,
    mae_tolerance_abs=0.05,
)

display(
    candidates_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
            "mean_ou_acc_edge_2": "{:.2%}",
            "mean_ou_acc_edge_3": "{:.2%}",
            "mean_ou_acc_edge_4": "{:.2%}",
        }
    )
)


[I 2026-04-10 19:21:06,406] A new study created in memory with name: xgb_total_points_mae


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-04-10 19:27:41,874] Trial 0 finished with value: 13.747924482374088 and parameters: {'max_depth': 2, 'min_child_weight': 18.346704707583235, 'gamma': 1.6970342240854253, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.5123279759064977, 'learning_rate': 0.011926786034588454, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 1.897469395521307, 'sample_weight_lambda': 0.0001422437941448231}. Best is trial 0 with value: 13.747924482374088.
[I 2026-04-10 19:39:38,762] Trial 1 finished with value: 13.673896738696179 and parameters: {'max_depth': 4, 'min_child_weight': 20.290108931287893, 'gamma': 0.3261777842309625, 'subsample': 0.8390562044499359, 'colsample_bytree': 0.4213034780854249, 'learning_rate': 0.012620826760486504, 'reg_alpha': 0.09307011182812809, 'reg_lambda': 15.258811505244246, 'sample_weight_lambda': 0.0010239553604139541}. Best is trial 1 with value: 13.673896738696179.
[I 2026-04-10 19:45:46,451] Trial 2 finished with value: 13.701769618555055 and parameters: {'

,trial,value_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,32,13.5432,16.9302,0.1457,57.01%,59.92%,61.25%,56.84%,133,50,2,5.121369,2.519867,0.739386,0.792048,0.054351,5.320959,3.036976,0.000450
1,43,13.5495,16.9389,0.1453,56.37%,59.67%,63.00%,68.65%,134,65,2,5.038961,2.969825,0.685155,0.738981,0.046716,5.935049,5.237797,0.000580
2,16,13.5559,16.9888,0.1392,56.58%,60.65%,63.99%,59.75%,95,65,3,41.317147,2.990696,0.737021,0.795393,0.039844,7.401220,7.666620,0.001919
3,13,13.5579,16.9540,0.1427,56.73%,62.21%,66.22%,72.56%,59,65,3,54.540601,2.273138,0.687146,0.796507,0.059463,19.640850,9.485937,0.000664
4,12,13.5641,16.9471,0.1443,56.86%,62.01%,65.59%,59.37%,161,91,3,33.396520,2.130883,0.765014,0.751839,0.029101,14.588800,10.389070,0.000463
5,17,13.5683,16.9789,0.1425,56.43%,60.64%,61.21%,57.16%,76,59,2,5.813154,2.933994,0.730501,0.799259,0.058556,5.826506,2.564178,0.002152
6,39,13.5692,16.9566,0.1429,56.79%,59.23%,64.35%,66.98%,101,87,4,16.523097,2.705004,0.715592,0.718208,0.038840,0.094586,1.991839,0.000323
7,33,13.5697,16.9791,0.1413,56.52%,61.61%,62.58%,47.87%,112,85,2,9.765650,2.563746,0.674690,0.764501,0.043180,10.986816,3.549771,0.000441
8,4,13.5704,17.0548,0.1326,57.25%,61.28%,60.07%,57.26%,107,104,3,27.769564,0.863656,0.554592,0.697083,0.040937,0.028904,9.594362,0.000551
9,42,13.5729,16.9928,0.1401,56.16%,61.37%,63.70%,65.89%,104,70,2,6.774553,2.910143,0.758000,0.797360,0.043588,4.271997,1.621381,0.002692


,trial,value_mae,mean_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,mae_cutoff,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,36,13.5828,13.5828,16.9570,0.1421,59.38%,59.20%,57.57%,53.51%,119,90,13.593189,2,23.271697,2.823680,0.594813,0.637537,0.051352,6.086536,8.167132,0.000473
1,4,13.5704,13.5704,17.0548,0.1326,57.25%,61.28%,60.07%,57.26%,107,104,13.593189,3,27.769564,0.863656,0.554592,0.697083,0.040937,0.028904,9.594362,0.000551
2,21,13.5827,13.5827,17.0196,0.1362,57.21%,59.91%,61.68%,53.18%,198,127,13.593189,3,40.705287,2.005049,0.795421,0.744804,0.022986,14.643734,9.748569,0.000663
3,32,13.5432,13.5432,16.9302,0.1457,57.01%,59.92%,61.25%,56.84%,133,50,13.593189,2,5.121369,2.519867,0.739386,0.792048,0.054351,5.320959,3.036976,0.000450
4,12,13.5641,13.5641,16.9471,0.1443,56.86%,62.01%,65.59%,59.37%,161,91,13.593189,3,33.396520,2.130883,0.765014,0.751839,0.029101,14.588800,10.389070,0.000463
5,39,13.5692,13.5692,16.9566,0.1429,56.79%,59.23%,64.35%,66.98%,101,87,13.593189,4,16.523097,2.705004,0.715592,0.718208,0.038840,0.094586,1.991839,0.000323
6,13,13.5579,13.5579,16.9540,0.1427,56.73%,62.21%,66.22%,72.56%,59,65,13.593189,3,54.540601,2.273138,0.687146,0.796507,0.059463,19.640850,9.485937,0.000664
7,16,13.5559,13.5559,16.9888,0.1392,56.58%,60.65%,63.99%,59.75%,95,65,13.593189,3,41.317147,2.990696,0.737021,0.795393,0.039844,7.401220,7.666620,0.001919
8,33,13.5697,13.5697,16.9791,0.1413,56.52%,61.61%,62.58%,47.87%,112,85,13.593189,2,9.765650,2.563746,0.674690,0.764501,0.043180,10.986816,3.549771,0.000441
9,17,13.5683,13.5683,16.9789,0.1425,56.43%,60.64%,61.21%,57.16%,76,59,13.593189,2,5.813154,2.933994,0.730501,0.799259,0.058556,5.826506,2.564178,0.002152


In [ ]:
def fit_and_predict_optuna_day_by_day(train_df, test_df):
    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model = fit_best_xgb_total_points(
        X_dev=X_train,
        y_dev=y_train,
        sample_weight_dates=train_df["GAME_DATE"],
        sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
        trial=best_trial_lexi,
        objective_name="reg:squarederror",
    )
    return model.predict(X_test)


day_by_day_optuna, day_by_day_optuna_predictions, day_by_day_optuna_daily, day_by_day_optuna_thresholds, day_by_day_optuna_profit = run_day_by_day_walk_forward_evaluation(
    label="Optuna-selected XGBoost",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_optuna_day_by_day,
    max_games=TRAIN_GAMES,
)

total_df = df_dev.tail(TRAIN_GAMES)


In [46]:
from nba_ou.modeling.modeling import ModelBundleMetadata, ModelInfo, TrainingMetrics

X_dev = total_df.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(total_df[TARGET_COL], errors="coerce")
sample_weight_dates_dev = total_df["GAME_DATE"]

best_model = fit_best_xgb_total_points(
    X_dev=X_dev,
    y_dev=y_dev,
    sample_weight_dates=sample_weight_dates_dev,
    sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
    trial=best_trial_lexi,
    objective_name="reg:squarederror",
)

y_pred_test_total = best_model.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)
ou_acc_edge_2 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=best_model,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)

df_to_train_split_rows = df_to_train.copy().tail(TRAIN_GAMES)

X_full = df_to_train_split_rows.drop(columns=EXCLUDE_COLS, errors="ignore")
y_full = pd.to_numeric(df_to_train_split_rows[TARGET_COL], errors="coerce")
sample_weight_dates_full = df_to_train_split_rows["GAME_DATE"]

production_model = fit_best_xgb_total_points(
    X_dev=X_full,
    y_dev=y_full,
    sample_weight_dates=sample_weight_dates_full,
    sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
    trial=best_trial_lexi,
    objective_name="reg:squarederror",
)

latest_training_date = pd.to_datetime(df_to_train_split_rows["GAME_DATE"]).max()
model_version = latest_training_date.strftime("%d_%m_%y")
model_name = f"three_seasons_xgb_total_points_{model_version}"

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version=model_version,
        model_type="three_seasons_total_points",
        prediction_source="three_seasons_xgb_total_points",
        training_code_tag="1.0",
    ),
    training_metrics=TrainingMetrics(
        best_params=best_trial_lexi.params,
        selected_trial_number=best_trial_lexi.number,
        mean_best_iteration=best_trial_lexi.user_attrs.get("mean_best_iteration"),
        median_best_iteration=best_trial_lexi.user_attrs.get("median_best_iteration"),
        cv_mae=float(best_trial_lexi.user_attrs.get("mean_mae", best_trial_lexi.value)),
        cv_rmse=best_trial_lexi.user_attrs.get("mean_rmse"),
        cv_ou_acc=best_trial_lexi.user_attrs.get("mean_ou_acc"),
        final_test_mae=float(mae),
        final_test_rmse=float(rmse),
        final_test_ou_acc=float(ou_acc),
        nan_threshold=nan_threshold,
        max_na_per_row=max_na_per_row,
        train_date_min=df_to_train_split_rows["GAME_DATE"].min().to_pydatetime(),
        train_date_max=df_to_train_split_rows["GAME_DATE"].max().to_pydatetime(),
        train_games=TRAIN_GAMES,
        sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    ),
)

model_path, meta_path = save_model_bundle(
    model=production_model,
    feature_names=list(X_full.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/3_seasons_weighted/",
    metadata=metadata,
)

print(
    f"Production model trained on {len(X_full)} rows using fixed n_estimators from median_best_iteration."
)
print("Saved model :", model_path)
print("Saved metadata:", meta_path)


Final test metrics
MSE: 320.73166
RMSE: 17.90898
MAE: 13.96882
OU_Betting_Accuracy: 49.83%
OU_Betting_Accuracy_Edge_2: 50.79%
OU_Betting_Accuracy_Edge_4: 55.88%


,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,298,100.0%,49.83%
1,1,204,68.5%,49.75%
2,2,128,43.0%,50.79%
3,3,72,24.2%,45.07%
4,4,35,11.7%,55.88%
5,5,16,5.4%,56.25%
6,6,7,2.3%,57.14%
7,7,3,1.0%,100.00%
8,8,1,0.3%,100.00%
9,9,0,0.0%,nan%


Production model trained on 2500 rows using fixed n_estimators from median_best_iteration.
Saved model : /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/3_seasons_weighted/three_seasons_xgb_total_points_08_04_26.json
Saved metadata: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/3_seasons_weighted/three_seasons_xgb_total_points_08_04_26.meta.json
